# Классификация ботов по cookie

Python 3.13.14, random_state=42. Open-source библиотеки из requirements.txt: pandas 3.0.6, numpy 2.5.3, scikit-learn 1.9.1, LightGBM 4.7.0. Они запускаются локально, внешние API не вызываются. Решение воспроизводится этим ноутбуком.

Метрика Precision при Recall 70% считается по cookie. Куки с одинаковым score обрабатываются одной группой. События берутся только внутри окна наблюдения. Валидация отсекается по времени с 17 апреля, потому что тестовые окна начинаются позже обучающих.


In [1]:
from pathlib import Path
import re

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

EVENT_TYPES = [
    "search_results_view",
    "item_view",
    "photo_swipe",
    "seller_page_view",
    "contact_phone_show",
    "contact_chat_open",
    "contact_message_sent",
    "favorite_add",
    "login",
    "captcha_shown",
]
BIGRAMS = [
    "search_results_view>item_view",
    "item_view>item_view",
    "item_view>search_results_view",
    "search_results_view>search_results_view",
    "item_view>photo_swipe",
    "photo_swipe>photo_swipe",
    "item_view>seller_page_view",
    "item_view>contact_phone_show",
    "item_view>favorite_add",
    "seller_page_view>contact_phone_show",
]
MILLION_CITIES = {
    "moskva", "sankt-peterburg", "novosibirsk", "ekaterinburg", "kazan",
    "nizhniy_novgorod", "chelyabinsk", "samara", "omsk", "rostov-na-donu",
    "ufa", "krasnoyarsk", "voronezh", "perm", "volgograd", "krasnodar",
}
ELECTRONICS = {"elektronika", "noutbuki", "telefony"}
AUTO = {"avtomobili", "zapchasti"}
REALTY = {"kvartiry_arenda", "kvartiry_prodazha"}
SCRAPER_UA = re.compile(
    r"HeadlessChrome|Scrapy|curl/|python-requests|python-urllib|Go-http-client|node-fetch",
    re.I,
)
CYRILLIC = re.compile(r"[а-яё]", re.I)
UA_FAMILIES = [
    "headless", "scrapy", "curl", "python", "gohttp", "node",
    "avito_android", "avito_ios", "yandex", "firefox",
    "chrome_mac", "chrome_win", "chrome_android", "safari_ios", "other",
]
PACKET_CRAWL = [
    "page_arith_run", "items_per_search", "item_uniq_x_new", "crawler_intensity",
    "item_pop_mean", "item_pop_min", "peer_item_pop_mean",
]
NEW_FEATURES = PACKET_CRAWL + [
    "fav_per_search", "contact_per_search", "events_after_login_share",
    "item_repeat_share", "login_x_locations", "login_x_events",
    "has_pointer", "event_entropy", "dt_bin_8_20", "power_user_trace",
]


def precision_at_recall(y_true, score, recall=0.70):
    # максимум precision среди порогов с recall не ниже 0.70
    # равные score нельзя разбить, группа отмечается целиком
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)
    n_pos = int(y_true.sum())
    if n_pos == 0:
        return float("nan")
    order = np.argsort(-score, kind="mergesort")
    y, s = y_true[order], score[order]
    tp = np.cumsum(y)
    k = np.arange(1, len(y) + 1)
    ends = np.r_[s[1:] != s[:-1], True]
    prec = tp[ends] / k[ends]
    rec = tp[ends] / n_pos
    ok = rec >= recall
    return float(prec[ok].max()) if ok.any() else 0.0


def normalize_platform(value):
    p = str(value).lower()
    if p in {"web", "desktop"}:
        return "web"
    if p == "android":
        return "android"
    if p in {"ios", "iphone"}:
        return "ios"
    return p


def events_in_window(events, meta):
    # условие задачи: event_ts в [window_start_ts, window_end_ts)
    ev = events.merge(meta[["cookie_id", "window_start_ts", "window_end_ts"]], on="cookie_id", how="inner")
    inside = (ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)
    return ev.loc[inside].drop(columns=["window_start_ts", "window_end_ts"])


def ua_family(ua):
    # сырая строка user-agent в модель не подаётся, только семейство
    u = ua or ""
    if "HeadlessChrome" in u:
        return "headless"
    if u.startswith("Scrapy"):
        return "scrapy"
    if u.startswith("curl"):
        return "curl"
    if u.startswith("python"):
        return "python"
    if u.startswith("Go-http"):
        return "gohttp"
    if u.startswith("node-fetch"):
        return "node"
    if u.startswith("Avito/") and "Android" in u:
        return "avito_android"
    if u.startswith("Avito/") and "iPhone" in u:
        return "avito_ios"
    if "YaBrowser" in u:
        return "yandex"
    if "Firefox" in u:
        return "firefox"
    if "Macintosh" in u:
        return "chrome_mac"
    if "Windows NT" in u:
        return "chrome_win"
    if "Android" in u and "Chrome" in u:
        return "chrome_android"
    if "iPhone" in u:
        return "safari_ios"
    return "other"


def _ua_flags(ua):
    s = ua.fillna("")
    chrome = s.str.extract(r"(?:HeadlessChrome|Chrome)/(\d+)", expand=False)
    avito = s.str.extract(r"Avito/(\d+)", expand=False)
    fam = s.map(ua_family)
    out = pd.DataFrame({
        "ua_scraper": s.str.contains(SCRAPER_UA, na=False).astype(np.int8),
        "ua_headless": s.str.contains("Headless", case=False, na=False).astype(np.int8),
        "ua_okhttp": s.str.contains("okhttp", case=False, na=False).astype(np.int8),
        "ua_avito_app": s.str.startswith("Avito/").astype(np.int8),
        "ua_yandex": s.str.contains("YaBrowser", na=False).astype(np.int8),
        "ua_firefox": s.str.contains("Firefox", na=False).astype(np.int8),
        "ua_linux": s.str.contains("X11; Linux", na=False).astype(np.int8),
        "ua_macos": s.str.contains("Macintosh", na=False).astype(np.int8),
        "ua_windows": s.str.contains("Windows NT", na=False).astype(np.int8),
        "ua_chrome_major": pd.to_numeric(chrome, errors="coerce"),
        "ua_avito_major": pd.to_numeric(avito, errors="coerce"),
    })
    for name in UA_FAMILIES:
        out[f"uaf_{name}"] = (fam == name).astype(np.int8)
    return out


def _group_slope(x, y):
    if len(x) < 3:
        return np.nan
    x = x.astype(float)
    y = y.astype(float)
    vx = np.var(x)
    if vx <= 1e-12:
        return 0.0
    return float(np.cov(x, y, bias=True)[0, 1] / vx)


def build_features(ev, meta):
    idx = meta["cookie_id"]
    ev = ev.copy()
    ev["platform_n"] = ev["platform"].map(normalize_platform)
    ev = ev.sort_values(["cookie_id", "event_ts"], kind="mergesort")
    ev["dt"] = ev.groupby("cookie_id", sort=False)["event_ts"].diff().dt.total_seconds()
    ev["hour"] = ev["event_ts"].dt.hour
    ev["minute"] = ev["event_ts"].dt.floor("min")
    ev["pos"] = ev.groupby("cookie_id", sort=False).cumcount()
    n_per = ev.groupby("cookie_id", sort=False)["pos"].transform("size")
    ev["rel_pos"] = ev["pos"] / n_per.clip(lower=1)
    ev["half"] = (ev["rel_pos"] >= 0.5).astype(np.int8)
    flags = _ua_flags(ev["user_agent"])
    ev = pd.concat([ev, flags], axis=1)
    g = ev.groupby("cookie_id", sort=False)

    feat = pd.DataFrame(index=pd.Index(idx, name="cookie_id"))
    feat["n_events"] = g.size()
    # дубликаты событий не удаляются, их число идёт признаком
    feat["n_dup_events"] = ev.duplicated(keep=False).groupby(ev["cookie_id"]).sum()
    feat["n_items"] = g["item_id"].nunique()
    feat["n_categories"] = g["item_category"].nunique()
    feat["n_locations"] = g["item_location"].nunique()
    feat["n_queries"] = g["search_query"].nunique()
    feat["n_ua"] = g["user_agent"].nunique()
    feat["n_platform_raw"] = g["platform"].nunique()
    feat["n_hours"] = g["hour"].nunique()

    counts = (
        ev.pivot_table(index="cookie_id", columns="event_name", values="eid", aggfunc="size", fill_value=0)
        .reindex(columns=EVENT_TYPES, fill_value=0)
        .astype(np.float32)
    )
    for name in EVENT_TYPES:
        feat[f"cnt_{name}"] = counts[name]
    n = feat["n_events"].replace(0, np.nan)
    for name in EVENT_TYPES:
        feat[f"share_{name}"] = feat[f"cnt_{name}"] / n

    feat["n_contact"] = feat["cnt_contact_phone_show"] + feat["cnt_contact_chat_open"] + feat["cnt_contact_message_sent"]
    feat["contact_per_view"] = feat["n_contact"] / (feat["cnt_item_view"] + 1)
    feat["view_per_search"] = feat["cnt_item_view"] / (feat["cnt_search_results_view"] + 1)
    feat["fav_per_view"] = feat["cnt_favorite_add"] / (feat["cnt_item_view"] + 1)
    feat["fav_per_search"] = feat["cnt_favorite_add"] / (feat["cnt_search_results_view"] + 1)
    feat["contact_per_search"] = feat["n_contact"] / (feat["cnt_search_results_view"] + 1)
    feat["photo_per_view"] = feat["cnt_photo_swipe"] / (feat["cnt_item_view"] + 1)
    feat["seller_per_view"] = feat["cnt_seller_page_view"] / (feat["cnt_item_view"] + 1)
    feat["has_login"] = (feat["cnt_login"] > 0).astype(np.int8)
    feat["has_search"] = (feat["cnt_search_results_view"] > 0).astype(np.int8)
    share_cols = [f"share_{name}" for name in EVENT_TYPES]
    share_mat = feat[share_cols].fillna(0).clip(lower=0).to_numpy(dtype=float)
    feat["event_entropy"] = -(np.where(share_mat > 0, share_mat * np.log(share_mat + 1e-12), 0.0)).sum(axis=1)

    search = ev.dropna(subset=["search_page"])
    if len(search):
        sg = search.groupby("cookie_id")
        feat["search_page_max"] = sg["search_page"].max()
        feat["search_page_mean"] = sg["search_page"].mean()
        feat["search_page_std"] = sg["search_page"].std()
        feat["deep_page10"] = (feat["search_page_max"] >= 10).astype(np.float32)
        feat["deep_page20"] = (feat["search_page_max"] >= 20).astype(np.float32)
        feat["query_uniq_ratio"] = sg["search_query"].nunique() / sg.size()
        feat["query_latin_share"] = search.assign(
            latin=~search["search_query"].str.contains(CYRILLIC, na=False)
        ).groupby("cookie_id")["latin"].mean()
        feat["search_page_slope"] = sg.apply(
            lambda d: _group_slope(d["pos"].to_numpy(), d["search_page"].to_numpy()),
            include_groups=False,
        )
        feat["search_page_mono_share"] = search.assign(
            up=search.groupby("cookie_id")["search_page"].diff() >= 0
        ).groupby("cookie_id")["up"].mean()
        page_jump = search.groupby("cookie_id")["search_page"].diff().abs()
        feat["page_jump_mean"] = page_jump.groupby(search["cookie_id"]).mean()
        feat["page_skip_share"] = page_jump.gt(1).groupby(search["cookie_id"]).mean()
        feat["n_search_pages"] = sg["search_page"].nunique()
        # длина серии страниц выдачи с шагом +1
        sp = search[["cookie_id", "search_page"]].copy()
        sp["inc"] = sp.groupby("cookie_id")["search_page"].diff().eq(1)
        sp["_chg"] = sp["inc"].ne(sp.groupby("cookie_id")["inc"].shift())
        sp["_run"] = sp.groupby("cookie_id")["_chg"].cumsum()
        run_len = sp.groupby(["cookie_id", "_run", "inc"]).size()
        if True in run_len.index.get_level_values("inc"):
            feat["page_arith_run"] = run_len.xs(True, level="inc", drop_level=False).groupby(level=0).max()
        else:
            feat["page_arith_run"] = 0.0
    feat["no_text_search"] = (feat["cnt_search_results_view"] == 0).astype(np.int8)

    loc = ev.dropna(subset=["item_location"])
    if len(loc):
        loc_counts = loc.groupby(["cookie_id", "item_location"]).size()
        loc_share = loc_counts / loc_counts.groupby(level=0).transform("sum")
        feat["loc_mode_share"] = loc_share.groupby(level=0).max()
        feat["loc_million_share"] = loc.assign(m=loc["item_location"].isin(MILLION_CITIES)).groupby("cookie_id")["m"].mean()
        feat["loc_msk_spb_share"] = loc.assign(
            m=loc["item_location"].isin({"moskva", "sankt-peterburg"})
        ).groupby("cookie_id")["m"].mean()
        loc_h = loc.groupby(["cookie_id", "half"])["item_location"].nunique().unstack(fill_value=0)
        if 0 in loc_h.columns and 1 in loc_h.columns:
            feat["loc_nuniq_first_half"] = loc_h[0]
            feat["loc_nuniq_second_half"] = loc_h[1]
            feat["loc_expansion"] = loc_h[1] - loc_h[0]

    cat = ev.dropna(subset=["item_category"])
    if len(cat):
        feat["cat_electronics_share"] = cat.assign(m=cat["item_category"].isin(ELECTRONICS)).groupby("cookie_id")["m"].mean()
        feat["cat_auto_share"] = cat.assign(m=cat["item_category"].isin(AUTO)).groupby("cookie_id")["m"].mean()
        feat["cat_realty_share"] = cat.assign(m=cat["item_category"].isin(REALTY)).groupby("cookie_id")["m"].mean()
        feat["cat_kids_share"] = cat.assign(m=cat["item_category"].eq("detskie_tovary")).groupby("cookie_id")["m"].mean()
        feat["cat_jobs_share"] = cat.assign(m=cat["item_category"].eq("rabota")).groupby("cookie_id")["m"].mean()
        cat_counts = cat.groupby(["cookie_id", "item_category"]).size()
        cat_share = cat_counts / cat_counts.groupby(level=0).transform("sum")
        feat["cat_mode_share"] = cat_share.groupby(level=0).max()

    seller = ev.dropna(subset=["seller_type"])
    if len(seller):
        feat["seller_pro_share"] = seller.assign(p=seller["seller_type"].eq("pro")).groupby("cookie_id")["p"].mean()

    items = ev.dropna(subset=["item_id"])
    if len(items):
        ig = items.groupby("cookie_id")["item_id"]
        feat["item_uniq_ratio"] = ig.nunique() / ig.size()

        def _item_uniq_slope(ids):
            seen = {}
            run = np.empty(len(ids), dtype=float)
            for i, v in enumerate(ids.to_numpy()):
                seen[v] = 1
                run[i] = len(seen)
            return _group_slope(np.arange(len(ids), dtype=float), run)

        feat["item_uniq_slope"] = items.groupby("cookie_id")["item_id"].agg(_item_uniq_slope)
        feat["item_repeat_share"] = 1.0 - feat["item_uniq_ratio"]
        feat["items_per_search"] = feat["n_items"] / (feat["cnt_search_results_view"] + 1)

    dt = ev.dropna(subset=["dt"])
    if len(dt):
        dg = dt.groupby("cookie_id")["dt"]
        feat["dt_median"] = dg.median()
        feat["dt_mean"] = dg.mean()
        feat["dt_std"] = dg.std()
        feat["dt_min"] = dg.min()
        feat["dt_cv"] = feat["dt_std"] / feat["dt_mean"].replace(0, np.nan)
        feat["fast_share"] = dt.assign(f=dt["dt"] < 5).groupby("cookie_id")["f"].mean()
        feat["very_fast_share"] = dt.assign(f=dt["dt"] < 2).groupby("cookie_id")["f"].mean()
        feat["idle_share"] = dt.assign(f=dt["dt"] > 180).groupby("cookie_id")["f"].mean()
        feat["dt_p10"] = dg.quantile(0.1)
        feat["dt_p90"] = dg.quantile(0.9)
        feat["dt_iqr"] = dg.quantile(0.75) - dg.quantile(0.25)
        for lo, hi, name in [
            (0, 1, "dt_bin_0_1"), (1, 3, "dt_bin_1_3"), (3, 10, "dt_bin_3_10"),
            (10, 30, "dt_bin_10_30"), (30, 120, "dt_bin_30_120"), (120, 1e9, "dt_bin_120p"),
        ]:
            feat[name] = dt.assign(f=(dt["dt"] >= lo) & (dt["dt"] < hi)).groupby("cookie_id")["f"].mean()
        feat["dt_bin_8_20"] = dt.assign(f=(dt["dt"] >= 8) & (dt["dt"] < 20)).groupby("cookie_id")["f"].mean()
        feat["dt_slope"] = dt.groupby("cookie_id").apply(
            lambda d: _group_slope(d["pos"].to_numpy(), d["dt"].to_numpy()),
            include_groups=False,
        )
        feat["n_same_ts"] = (
            ev.assign(_k=1).groupby(["cookie_id", "event_ts"])["_k"].transform("size").gt(1).groupby(ev["cookie_id"]).mean()
        )

    feat["hour_span"] = g["hour"].agg(lambda s: int(s.max() - s.min()) if len(s) else 0)
    feat["night_share"] = ev.assign(n=ev["hour"].isin([0, 1, 2, 3, 4])).groupby("cookie_id")["n"].mean()
    feat["max_per_hour"] = ev.groupby(["cookie_id", "hour"]).size().groupby("cookie_id").max()
    feat["max_per_minute"] = ev.groupby(["cookie_id", "minute"]).size().groupby("cookie_id").max()
    span_sec = g["event_ts"].agg(lambda s: (s.max() - s.min()).total_seconds())
    feat["session_span_sec"] = span_sec
    feat["events_per_min"] = feat["n_events"] / (span_sec / 60.0 + 1e-3)

    ev["bin4"] = (ev["hour"] // 6).astype(np.int8)
    bin_counts = ev.groupby(["cookie_id", "bin4"]).size().unstack(fill_value=0)
    for b in range(4):
        feat[f"share_bin{b}"] = bin_counts.get(b, 0) / n
    half_n = ev.groupby(["cookie_id", "half"]).size().unstack(fill_value=0)
    if 0 in half_n.columns and 1 in half_n.columns:
        feat["second_half_share"] = half_n[1] / (half_n[0] + half_n[1] + 1e-6)

    prev = ev.groupby("cookie_id")["event_name"].shift()
    bg = prev + ">" + ev["event_name"]
    bg_ct = pd.crosstab(ev["cookie_id"], bg).reindex(columns=BIGRAMS, fill_value=0).astype(np.float32)
    n_trans = ev.groupby("cookie_id").size() - 1
    for name in BIGRAMS:
        short = name.replace("_", "")[:28]
        feat[f"bg_{short}"] = bg_ct[name]
        feat[f"bgshare_{short}"] = bg_ct[name] / n_trans.clip(lower=1)

    # pointer есть только на web, на mobile медиану не подставляем
    ptr = ev.dropna(subset=["pointer_x", "pointer_y"]).copy()
    feat["n_pointer"] = ptr.groupby("cookie_id").size()
    feat["pointer_missing_share"] = 1.0 - feat["n_pointer"].fillna(0) / n
    feat["has_pointer"] = (feat["n_pointer"].fillna(0) > 0).astype(np.int8)
    if len(ptr):
        pg = ptr.groupby("cookie_id")
        feat["ptr_x_std"] = pg["pointer_x"].std()
        feat["ptr_y_std"] = pg["pointer_y"].std()
        feat["ptr_x_mean"] = pg["pointer_x"].mean()
        feat["ptr_y_mean"] = pg["pointer_y"].mean()
        uniq_xy = ptr.groupby("cookie_id", group_keys=False)[["pointer_x", "pointer_y"]].apply(
            lambda d: d.drop_duplicates().shape[0]
        )
        feat["ptr_uniq_xy"] = uniq_xy
        feat["ptr_uniq_ratio"] = feat["ptr_uniq_xy"] / feat["n_pointer"]
        ptr["pdist"] = np.hypot(ptr.groupby("cookie_id")["pointer_x"].diff(), ptr.groupby("cookie_id")["pointer_y"].diff())
        pdg = ptr.dropna(subset=["pdist"]).groupby("cookie_id")["pdist"]
        feat["ptr_jump_median"] = pdg.median()
        feat["ptr_jump_std"] = pdg.std()
        feat["ptr_jump_mean"] = pdg.mean()

    extra = pd.DataFrame({col: g[col].mean() for col in flags.columns})
    extra["any_ua_scraper"] = g["ua_scraper"].max()
    plat = g["platform_n"].agg(lambda s: s.mode().iloc[0] if len(s) else "unknown")
    extra["plat_web"] = plat.eq("web").astype(np.int8)
    extra["plat_android"] = plat.eq("android").astype(np.int8)
    extra["plat_ios"] = plat.eq("ios").astype(np.int8)

    m = meta.set_index("cookie_id")
    age = (m["window_start_ts"] - m["cookie_created_at"]).dt.total_seconds() / 86400.0
    extra["cookie_age_days"] = age
    extra["cookie_age_log"] = np.log1p(age.clip(lower=0))
    extra["cookie_is_new_1d"] = (age < 1).astype(np.int8)
    extra["cookie_is_new_7d"] = (age < 7).astype(np.int8)
    extra["events_per_age"] = feat["n_events"] / (age.clip(lower=0) + 1.0)
    extra["pages_per_age"] = feat.get("search_page_max", 0) / (age.clip(lower=0) + 1.0)
    extra["old_and_busy"] = (age >= 14).astype(float) * np.log1p(feat["n_events"])
    extra["young_and_busy"] = (age < 3).astype(float) * np.log1p(feat["n_events"])
    extra["busy_logged_in"] = feat["n_events"] * feat["has_login"]
    extra["login_x_locations"] = feat["has_login"] * feat["n_locations"].fillna(0)
    extra["login_x_events"] = feat["has_login"] * feat["n_events"].fillna(0)
    first_login = ev.loc[ev["event_name"].eq("login")].groupby("cookie_id")["pos"].min()
    login_pos = ev["cookie_id"].map(first_login)
    extra["events_after_login_share"] = (ev["pos"] > login_pos).groupby(ev["cookie_id"]).mean()
    extra["run_len_max"] = (
        ev.assign(_chg=ev["event_name"].ne(ev.groupby("cookie_id")["event_name"].shift()))
        .assign(_run=lambda d: d.groupby("cookie_id")["_chg"].cumsum())
        .groupby(["cookie_id", "_run"]).size().groupby("cookie_id").max()
    )

    feat = pd.concat([feat, extra], axis=1).reindex(idx)
    count_like = [c for c in feat.columns if c.startswith("cnt_") or c.startswith("n_") or c.startswith("bg_")]
    feat[count_like] = feat[count_like].fillna(0)
    feat["n_events"] = feat["n_events"].fillna(0)
    feat["has_pointer"] = feat["has_pointer"].fillna(0).astype(np.int8)
    if "page_arith_run" not in feat.columns:
        feat["page_arith_run"] = 0.0
    feat["page_arith_run"] = feat["page_arith_run"].fillna(0)
    feat["events_after_login_share"] = feat["events_after_login_share"].fillna(0)
    if "item_uniq_ratio" not in feat.columns:
        feat["item_uniq_ratio"] = np.nan
    if "ptr_x_std" not in feat.columns:
        feat["ptr_x_std"] = np.nan
    feat["item_uniq_x_new"] = feat["item_uniq_ratio"].fillna(0) * feat["cookie_is_new_7d"].fillna(0)
    feat["crawler_intensity"] = (
        (1.0 - feat["has_login"].fillna(0))
        * np.log1p(feat["search_page_max"].fillna(0))
        * np.log1p(feat["n_locations"].fillna(0))
        / (feat["cookie_age_days"].clip(lower=0).fillna(30) + 1.0)
    )
    ptr_web = feat["ptr_x_std"].where(feat["plat_web"].fillna(0).eq(1))
    feat["power_user_trace"] = (
        feat["has_login"].fillna(0)
        * np.log1p(feat["n_contact"].fillna(0))
        * np.log1p(feat["cookie_age_days"].clip(lower=0).fillna(0))
        * np.log1p(ptr_web.fillna(0) * feat["has_pointer"].fillna(0))
    )
    return feat.copy().reset_index()


def feature_columns(frame):
    return [c for c in frame.columns if c != "cookie_id"]


def present_cols(frame, names):
    have = set(frame.columns)
    return [c for c in names if c in have]


def add_peer_ranks(feat, meta):
    out = feat.copy()
    day = meta.set_index("cookie_id").loc[out["cookie_id"], "window_start_ts"].to_numpy()
    tmp = out.copy()
    tmp["_day"] = day
    for col in ["n_events", "n_locations", "n_items", "search_page_max", "max_per_hour", "events_per_min", "dt_median", "item_pop_mean"]:
        if col in tmp.columns:
            out[f"peer_{col}"] = tmp.groupby("_day")[col].rank(pct=True, method="average")
    return out


def attach_item_pop(feat, ev, pop):
    # частота объявления без целевой метки: сколько разных cookie его открывали
    out = feat.drop(columns=[c for c in ("item_pop_mean", "item_pop_min") if c in feat.columns], errors="ignore")
    e = ev.dropna(subset=["item_id"]).copy()
    if len(e) == 0:
        out["item_pop_mean"] = 0.0
        out["item_pop_min"] = 0.0
        return out
    e["_pop"] = e["item_id"].map(pop).fillna(0)
    agg = e.groupby("cookie_id").agg(item_pop_mean=("_pop", "mean"), item_pop_min=("_pop", "min"))
    out = out.merge(agg, left_on="cookie_id", right_index=True, how="left")
    out["item_pop_mean"] = np.log1p(out["item_pop_mean"].fillna(0))
    out["item_pop_min"] = np.log1p(out["item_pop_min"].fillna(0))
    return out


def pos_weight(y):
    pos = max(int(y.sum()), 1)
    return (len(y) - pos) / pos


def make_lgbm(y, n_estimators=500, random_state=RANDOM_STATE, num_leaves=31, min_child_samples=40, colsample_bytree=0.7):
    return lgb.LGBMClassifier(
        n_estimators=n_estimators,
        learning_rate=0.04,
        num_leaves=num_leaves,
        min_child_samples=min_child_samples,
        subsample=0.8,
        colsample_bytree=colsample_bytree,
        reg_lambda=1.0,
        scale_pos_weight=pos_weight(y),
        random_state=random_state,
        verbose=-1,
    )


def fit_lgbm(X_fit, y_fit, X_va, cols, w=None, **lgbm_kw):
    clf = make_lgbm(y_fit, **lgbm_kw)
    clf.fit(X_fit[cols], y_fit, sample_weight=w)
    return clf.predict_proba(X_va[cols])[:, 1], clf


def lgbm_seed_bag(X_fit, y_fit, X_va, cols, seeds=(42, 0, 7, 13, 99), w=None, **lgbm_kw):
    preds = []
    models = []
    for seed in seeds:
        clf = make_lgbm(y_fit, random_state=seed, **lgbm_kw)
        clf.fit(X_fit[cols], y_fit, sample_weight=w)
        preds.append(clf.predict_proba(X_va[cols])[:, 1])
        models.append(clf)
    return np.mean(np.vstack(preds), axis=0), models


DATA = Path("data")
train = pd.read_csv(DATA / "train.csv", parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"])
test = pd.read_csv(DATA / "test.csv", parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"])
events = pd.read_csv(DATA / "events.csv", parse_dates=["event_ts"])

print(train.shape, test.shape, events.shape, "bot_rate", round(train.target.mean(), 4))
print("train", train.window_start_ts.min().date(), "-", train.window_end_ts.max().date())
print("test ", test.window_start_ts.min().date(), "-", test.window_end_ts.max().date())


(11091, 5) (4909, 4) (328905, 14) bot_rate 0.0811
train 2026-04-06 - 2026-04-20
test  2026-04-20 - 2026-04-27


Обучающая выборка покрывает окна с 6 по 19 апреля, тестовая с 20 по 26 апреля. Случайное разбиение train здесь не подходит, потому что тест лежит позже по времени.

## Исходные гипотезы

Готовых поведенческих признаков в таблицах нет, их нужно собрать из events.csv. После user_agent многие колонки выглядят пустыми, поэтому сначала проверяется полнота и решается, какие поля включать в модель.

1. Необходимо посмотреть полноту колонок после агента. Если пропуски зависят от типа события, поле для этого типа не заполняется, и это нужно учитывать при сборке признаков.
2. Необходимо сопоставить eid и event_name и исключить выбросы по кодам, если они есть.
3. Нужно разобрать связь platform и user-agent, в том числе на куках без входа в аккаунт.
4. Нужно выделить популярные локации и категории. По локациям ожидались города-миллионники, Москва с областью и Санкт-Петербург с Ленинградской областью. По категориям ожидались электроника, ноутбуки, автомобили, квартиры, аренда, детские товары, работа и запчасти. Имеет смысл проверить дубликаты написания и не объединять близкие слаги вроде электроники и ноутбуков.
5. Подозрительны куки без текстового поискового запроса или с запросом на английском или транслите.
6. Подозрительны куки, с которых с одной локации, одного компьютера или одного агента идёт много запросов.
7. Подозрительны куки, у которых почти не меняются координаты курсора.


In [2]:
print(events.groupby(["eid", "event_name"]).size().reset_index(name="n").to_string(index=False))
print("na share")
print(events.isna().mean().round(3).to_string())
cap = events.loc[events.event_name.eq("captcha_shown")].merge(
    train[["cookie_id", "window_start_ts", "window_end_ts"]], on="cookie_id", how="inner"
)
in_w = (cap.event_ts >= cap.window_start_ts) & (cap.event_ts < cap.window_end_ts)
print("captcha_shown на train-куках", len(cap), "внутри окна", int(in_w.sum()),
      "на test-куках", int((events.event_name.eq("captcha_shown") & events.cookie_id.isin(test.cookie_id)).sum()))


 eid           event_name      n
 100  search_results_view 100402
 200            item_view 120817
 210          photo_swipe  36517
 220     seller_page_view  17403
 300   contact_phone_show  11316
 301    contact_chat_open   6125
 303 contact_message_sent   3081
 400         favorite_add  19049
 500                login   6267
 900        captcha_shown   7928
na share
cookie_id        0.000
event_ts         0.000
eid              0.000
event_name       0.000
platform         0.000
user_agent       0.000
item_id          0.348
item_category    0.100
item_location    0.072
seller_type      0.407
search_query     0.695
search_page      0.695
pointer_x        0.670
pointer_y        0.670
captcha_shown на train-куках 7928 внутри окна 0 на test-куках 0


In [3]:
# полнота полей после user_agent внутри типа события
cols = ["item_id", "item_category", "item_location", "seller_type", "search_query", "search_page", "pointer_x"]
print(events.groupby("event_name")[cols].agg(lambda s: s.isna().mean()).round(2).to_string())


                      item_id  item_category  item_location  seller_type  search_query  search_page  pointer_x
event_name                                                                                                    
captcha_shown             1.0           1.00           1.00         1.00           1.0          1.0       0.73
contact_chat_open         0.0           0.06           0.03         0.08           1.0          1.0       0.69
contact_message_sent      0.0           0.06           0.03         0.09           1.0          1.0       0.68
contact_phone_show        0.0           0.06           0.03         0.09           1.0          1.0       0.68
favorite_add              0.0           0.06           0.03         0.09           1.0          1.0       0.65
item_view                 0.0           0.06           0.03         0.09           1.0          1.0       0.67
login                     1.0           1.00           1.00         1.00           1.0          1.0       0.65
p

## Обработка данных

Пропуски не заполняются общей медианой, потому что пустое значение часто связано с типом события или платформой.

Дубликаты строк в events не удаляются. Их число на куку сохраняется как признак n_dup_events.

В признаки попадают только события с event_ts внутри интервала window_start_ts и window_end_ts. Строки вне окна отбрасываются.

События captcha_shown в train находятся вне окна, в test их нет, поэтому капча в признаки не входит.

Коды eid и названия event_name совпадают один к одному, типов десять, неизвестных кодов нет. В модель передаётся название события.

Значения platform приведены к трём группам web, android и ios. Варианты web, WEB, desktop и пара ios или iphone после нормализации дают одну платформу на куку.

Сырая строка user-agent в модель не подаётся из-за высокой кардинальности. Строка разбирается на семейство браузера или приложения и на флаги HeadlessChrome, Scrapy, curl, python-requests, Go-http-client, node-fetch. Такие агенты почти всегда принадлежат ботам и покрывают небольшую долю положительного класса. Основная часть размеченных ботов ходит с обычным Chrome.

Координаты курсора заполнены только на web. На android и ios колонка пустая, и подставлять туда медиану с десктопа нельзя, иначе сессия без курсора становится похожа на человека с мышью. В модели pointer на mobile остаётся пустым, на web добавляется флаг наличия координат.

Target encoding по item_id не используется. Объявление обычно связано с одной cookie, и такая кодировка переносит целевую метку. Частота объявления считается без лейбла как число различных cookie, которые его открывали, по событиям train и test вместе.


In [4]:
ev = events.merge(train[["cookie_id", "target", "window_start_ts", "window_end_ts"]], on="cookie_id")
ev = ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)].copy()
ev["plat"] = ev.platform.map(normalize_platform)
ev["uaf"] = ev.user_agent.fillna("").map(ua_family)

print(pd.crosstab(ev.platform, ev.plat))
print("платформ на куку", ev.groupby("cookie_id").plat.nunique().value_counts().to_dict())

ck = train.set_index("cookie_id")[["target"]].copy()
ck["plat"] = ev.groupby("cookie_id").plat.first()
ck["uaf"] = ev.groupby("cookie_id").uaf.first()
ck["has_login"] = ev.event_name.eq("login").groupby(ev.cookie_id).any().astype(int)
print("bot rate platform")
print(ck.groupby("plat").target.agg(["mean", "count"]))
print("bot rate ua (top)")
print(ck.groupby("uaf").target.agg(["mean", "count"]).sort_values("mean", ascending=False).head(12))
print("login vs bot", ck.groupby("has_login").target.mean().round(4).to_dict())


plat      android   ios    web
platform                      
ANDROID     26209     0      0
Android     25893     0      0
IOS             0  2634      0
WEB             0     0  27431
Web             0     0  27470
android     25889     0      0
desktop         0     0  27855
iOS             0  2626      0
ios             0  2576      0
iphone          0  2602      0
web             0     0  27251
платформ на куку {1: 11091}
bot rate platform
             mean  count
plat                    
android  0.062024   4595
ios      0.044905    579
web      0.099375   5917
bot rate ua (top)
                   mean  count
uaf                           
node           0.272727     33
headless       0.262238    286
scrapy         0.257143     35
curl           0.250000     32
python         0.189655     58
gohttp         0.166667     42
avito_android  0.102623   1754
chrome_win     0.092539   1394
firefox        0.087291   1432
chrome_mac     0.082669   1379
yandex         0.082382   1226
safar

In [5]:
loc = ev.dropna(subset=["item_location"])
cat = ev.dropna(subset=["item_category"])
print("top loc")
print(loc.item_location.value_counts().head(12))
print("top cat")
print(cat.item_category.value_counts().head(12))

ck["n_loc"] = loc.groupby("cookie_id").item_location.nunique()
print("n_loc vs bot")
print(
    ck.assign(g=pd.cut(ck.n_loc.fillna(0), [-0.1, 1, 3, 8, 999], labels=["1", "2-3", "4-8", "9+"]))
    .groupby("g", observed=False)
    .target.agg(["mean", "count"])
)

search = ev.loc[ev.event_name.eq("search_results_view")]
ck["n_search"] = search.groupby("cookie_id").size()
ck["page_max"] = search.groupby("cookie_id").search_page.max()
ck["latin_q"] = search.assign(
    latin=~search.search_query.fillna("").str.contains(r"[а-яё]", case=False)
).groupby("cookie_id").latin.mean()
print("нет текстового поиска, bot rate", round(ck.loc[ck.n_search.fillna(0).eq(0), "target"].mean(), 4))
print("latin share by target", ck.groupby("target").latin_q.mean().round(3).to_dict())
print("page>=20 bot rate", round(ck.loc[ck.page_max.fillna(0).ge(20), "target"].mean(), 4))

ptr = ev.dropna(subset=["pointer_x", "pointer_y"])
print("pointer share by plat", ev.assign(p=ev.pointer_x.notna()).groupby("plat").p.mean().round(3).to_dict())
ck["ptr_std"] = ptr.groupby("cookie_id").pointer_x.std()
print("ptr_x std median by target (web)", ck.loc[ck.plat.eq("web")].groupby("target").ptr_std.median().to_dict())
print("share cookie с >1 уникальным x или y", round((ptr.groupby("cookie_id")[["pointer_x", "pointer_y"]].nunique().max(axis=1) > 1).mean(), 3))


top loc
item_location
sankt-peterburg    24676
novosibirsk        23533
moskva             23169
ekaterinburg       23098
ryazan              2933
omsk                2803
barnaul             2787
astrahan            2774
kazan               2763
rostov-na-donu      2758
krasnodar           2740
izhevsk             2732
Name: count, dtype: int64
top cat
item_category
avtomobili           13158
mebel                12688
kvartiry_prodazha    12531
hobbi                12514
odezhda              12453
uslugi               12213
noutbuki             12178
elektronika          12162
telefony             12137
bytovaya_tehnika     12108
detskie_tovary       12063
rabota               11849
Name: count, dtype: int64
n_loc vs bot
         mean  count
g                   
1    0.017576   1081
2-3  0.033412   2963
4-8  0.079008   4354
9+   0.162273   2693
нет текстового поиска, bot rate 0.0402
latin share by target {0: 0.187, 1: 0.174}
page>=20 bot rate 0.7913
pointer share by plat {'android': 

## Результаты разведочного анализа

Пропуски после user_agent зависят от типа события. Категория и город заполнены на карточках объявлений, поисковый запрос и страница выдачи появляются на search_results_view, координаты курсора есть только на web.

Коды событий согласованы. Есть десять пар eid и event_name, неизвестных значений нет.

На куках без login доля ботов немного выше, сам факт входа слабый признак. Приложение Avito ближе к человеческому трафику. Явные scraper-агенты почти целиком относятся к ботам и покрывают малую долю разметки, поэтому дальше нужны поведенческие признаки.

По локациям исходное ожидание в целом подтверждается: сверху Москва, Санкт-Петербург, Новосибирск, Екатеринбург. Областей в слагах нет, дублей написания не видно. Категории elektronika, noutbuki и telefony хранятся отдельно, как аренда и продажа квартир, автомобили и запчасти. Эти слаги не объединялись.

Гипотеза про отсутствие текста и латиницу не подтвердилась. Доля ботов там близка к средней, латиница часто оказывается названием бренда вроде iphone 13 или toyota camry.

Гипотеза про один компьютер и поток запросов сработала иначе, чем ожидалось. Один user-agent на куку встречается у обычных пользователей. Сильнее разделяет ширина обхода: при одной локации доля ботов около 2%, при большом числе городов она заметно выше. Страницы выдачи от 20 и выше содержат очень высокую долю ботов. Большое число событий полезно, при этом тот же объём характерен для людей с длинной сессией, и это видно позже на PCA.

Неподвижный курсор почти не встречается, координаты обычно меняются. У ботов короче прыжки и меньше разброс pointer_x. На мобильных платформах курсора нет, поэтому седьмая гипотеза в исходном виде для android и ios неприменима.


In [6]:
# счётчики событий, страницы, dt, семейство ua, pointer на web, возраст куки, peer-ranks внутри дня
# пакет crawl после pca: серия страниц +1, items на поиск, item_uniq у новой куки, интенсивность, item_pop без лейбла
ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)
pop = (
    pd.concat([ev_tr[["cookie_id", "item_id"]], ev_te[["cookie_id", "item_id"]]], ignore_index=True)
    .dropna(subset=["item_id"])
    .groupby("item_id")["cookie_id"]
    .nunique()  # без лейбла, train+test события
)
Xtr = add_peer_ranks(attach_item_pop(build_features(ev_tr, train), ev_tr, pop), train)
Xte = add_peer_ranks(attach_item_pop(build_features(ev_te, test), ev_te, pop), test)
y = train.target.to_numpy(dtype=int)
is_valid = train.window_start_ts.ge("2026-04-17").to_numpy()

new = set(present_cols(Xtr, NEW_FEATURES))
base = [c for c in feature_columns(Xtr) if c not in new]
cols = base + present_cols(Xtr, PACKET_CRAWL)
print("cols", len(cols), "base", len(base), "fit", int((~is_valid).sum()), "valid", int(is_valid.sum()))


/var/folders/rx/t02l87nj2f55krbztd18_lg00000gn/T/ipykernel_44420/1488587404.py:346: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  feat[f"share_bin{b}"] = bin_counts.get(b, 0) / n
/var/folders/rx/t02l87nj2f55krbztd18_lg00000gn/T/ipykernel_44420/1488587404.py:349: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  feat["second_half_share"] = half_n[1] / (half_n[0] + half_n[1] + 1e-6)
/var/folders/rx/t02l87nj2f55krbztd18_lg00000gn/T/ipykernel_44420/1488587404.py:357: PerformanceWarning: DataFrame is highly fragmented.  This is usually t

cols 180 base 173 fit 9140 valid 1951


/var/folders/rx/t02l87nj2f55krbztd18_lg00000gn/T/ipykernel_44420/1488587404.py:424: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  feat["item_uniq_x_new"] = feat["item_uniq_ratio"].fillna(0) * feat["cookie_is_new_7d"].fillna(0)
/var/folders/rx/t02l87nj2f55krbztd18_lg00000gn/T/ipykernel_44420/1488587404.py:425: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  feat["crawler_intensity"] = (
/var/folders/rx/t02l87nj2f55krbztd18_lg00000gn/T/ipykernel_44420/1488587404.py:432: PerformanceWarning: DataFrame is highly fragmented.  This is us

## PCA

PCA нужен, чтобы понять, разделяются ли cookie в пространстве поведения на явных парсеров, обычных пользователей и промежуточную группу.

На train берутся числовые поведенческие колонки без target encoding по item. После стандартизации строится PCA и K-means на пять кластеров по первым восьми компонентам. На плоскости первых двух компонент объясняется около 22% дисперсии, поэтому это проекция. Для PCA пропуски pointer заполняются медианой, чтобы алгоритм отработал. В классификаторе на mobile эти пропуски сохраняются.


In [7]:
num = [c for c in base if pd.api.types.is_numeric_dtype(Xtr[c])]
Z = SimpleImputer(strategy="median").fit_transform(Xtr[num])
Z = StandardScaler().fit_transform(Z)
pca = PCA(n_components=8, random_state=RANDOM_STATE)
P = pca.fit_transform(Z)
print("var PC1+PC2", round(pca.explained_variance_ratio_[:2].sum(), 3), "pc1", num[int(np.abs(pca.components_[0]).argmax())], "pc2", num[int(np.abs(pca.components_[1]).argmax())])

km = KMeans(n_clusters=5, n_init=10, random_state=RANDOM_STATE)
lab = km.fit_predict(P)
tmp = pd.DataFrame({
    "c": lab,
    "y": y,
    "n_events": Xtr.n_events.to_numpy(),
    "n_loc": Xtr.n_locations.fillna(0).to_numpy(),
    "page": Xtr.search_page_max.fillna(0).to_numpy(),
    "age": Xtr.cookie_age_days.fillna(0).to_numpy(),
    "login": Xtr.has_login.fillna(0).to_numpy(),
    "web": Xtr.plat_web.fillna(0).to_numpy(),
    "ptr": Xtr.ptr_x_std.to_numpy(),
})
g = tmp.groupby("c").agg(
    n=("y", "size"),
    bots=("y", "sum"),
    rate=("y", "mean"),
    events=("n_events", "median"),
    loc=("n_loc", "median"),
    page=("page", "median"),
    age=("age", "median"),
    login=("login", "mean"),
    web=("web", "mean"),
    ptr=("ptr", "median"),
).sort_values("rate", ascending=False)
g["share_of_bots"] = g.bots / g.bots.sum()
print(g.round(3).to_string())


var PC1+PC2 0.202 pc1 n_events pc2 peer_events_per_min
      n  bots   rate  events   loc  page     age  login    web      ptr  share_of_bots
c                                                                                     
4   280    71  0.254    14.5   6.0   5.0  48.848  0.225  1.000  547.211          0.079
0  1023   210  0.205    54.0  16.0   9.0  53.077  0.605  0.609  544.630          0.234
3  3597   325  0.090    22.0   8.0   5.0  59.508  0.362  0.540  547.629          0.362
2  2269   119  0.052     8.0   3.0   2.0  61.987  0.176  0.519  542.811          0.132
1  3922   174  0.044     7.0   3.0   2.0  60.528  0.135  0.483  542.670          0.194


Один кластер содержит около 310 cookie и почти 73% ботов внутри себя, это примерно четверть всех ботов. Для него характерны новая кука возрастом около полутора дней, web, много событий и городов, глубокая выдача, узкий pointer, почти нет login и избранного. Базовая модель и так ставит этот кластер наверх рейтинга, отдельный детектор почти не двигает Precision при Recall 70%.

Рядом лежит большой web-кластер с долей ботов около 9%, и в нём сосредоточено больше трети всех ботов. Это обычный Chrome без логина и со средней активностью. На этом участке метрика чувствительна к порядку в середине списка.

Есть кластер с похожим объёмом и географией. У него старая кука, логин, избранное, контакты и широкий pointer. Это активные пользователи, и на них приходятся ложные срабатывания.

Отдельно тихий web и mobile с малым числом событий и одной-тремя локациями. Там сосредоточены пропуски, которые суточным окном почти не описываются.

Первая компонента связана с объёмом (события, города, страницы выдачи), вторая с темпом (события в минуту и возраст куки).



## Модели

Все оценки ниже получены на валидации с 17 апреля по метрике Precision при Recall 70%.

Константа даёт 0.08, один признак n_events около 0.12, набор правил на Headless, глубокую выдачу, много городов и новую куку 0.16. Линейные модели linreg, logreg и svm дают 0.38-0.43 при ROC около 0.89. Класс отделяется, при полноте 70% линейной границы недостаточно.

Xgboost 0.61, Catboost 0.63, Random Forest 0.65. LightGBM на поведенческих признаках 0.713, и этот результат используется как базовая точка. Среднее LightGBM, XGBoost и CatBoost даёт 0.64, потому что усреднение сглаживает верх рейтинга.

Сжатие через Lasso и Grey Wolf, MLP как экстрактор, self-training и правила поверх бустинга оказались ниже полного LightGBM. Изображений, физических уравнений и текстового корпуса в задаче нет, поэтому VGG, PINN и BERT не применялись.

Target encoding по item_id переносит лейбл, валидация падает примерно до 0.37. После этого сабмит не выбирается автоматически по таблице сравнения.

На пороге базового LightGBM примерно 114 true positive, 46 false positive и 46 false negative. Кластер с высокой долей ботов уже весь в true positive, ошибки остаются в середине: mix и mobile дают ложные срабатывания, короткие сессии дают пропуски. Дальнейшие признаки добавлялись пакетами под эти ошибки.

Пакет catalog-crawl поднимает метрику до 0.830 за счёт длины серии страниц с шагом +1, числа объявлений на поиск и частоты item без лейбла. Human-trace даёт 0.727, pointer-flag и power-crawler 0.709, dt_entropy 0.667 и этот пакет ухудшает результат. Все новые признаки сразу дают 0.818, ниже чем catalog-crawl отдельно.

Двухступенчатая схема с правилом острова сверху и LightGBM снизу даёт 0.857 при более низком PR-AUC, чем у bag. Раздельные модели web и mobile дают 0.806 по целевой метрике. Сетка параметров подбиралась по Precision при Recall 70%, без оптимизации logloss. Лучшие значения num_leaves 48, min_child_samples 60, colsample_bytree 0.9.

Итоговый вариант это bag из пяти сидов на признаках catalog-crawl, 0.857 на валидации. Калибровка вероятностей и случайный шум на совпадающие score не использовались, потому что метрика зависит от порядка и от групп равных значений.


In [8]:
X_fit, X_va = Xtr.loc[~is_valid], Xtr.loc[is_valid]
y_fit, y_va = y[~is_valid], y[is_valid]
kw = dict(num_leaves=48, min_child_samples=60, colsample_bytree=0.9)

p_base, _ = fit_lgbm(X_fit, y_fit, X_va, base)
p_crawl, _ = fit_lgbm(X_fit, y_fit, X_va, cols)
p_bag, _ = lgbm_seed_bag(X_fit, y_fit, X_va, cols, **kw)

for name, p in [("якорь behav+peers", p_base), ("+catalog_crawl", p_crawl), ("bag 48/60/0.9", p_bag)]:
    print(f"{name:22} P@R70={precision_at_recall(y_va, p):.4f}")


якорь behav+peers      P@R70=0.7125
+catalog_crawl         P@R70=0.8296
bag 48/60/0.9          P@R70=0.8571


Гипотезы 1-4 и 6 вошли в признаки и в правила обработки. Пятая гипотеза не подтвердилась. Седьмая в формулировке неподвижного курсора тоже, при этом разброс pointer на web остался в модели.

PCA сместил акцент с явных парсеров, которые и так стоят наверху рейтинга, на ранжирование середины. Гибридные схемы и target encoding по item отброшены. В сабмите LightGBM, признаки catalog-crawl и усреднение по пяти сидам.

Короткие мобильные сессии остаются среди пропусков, потому что за сутки по ним мало сигнала. Дальнейшая подстройка под три дня валидации повышает риск переобучения.


In [9]:
p_test, _ = lgbm_seed_bag(Xtr, y, Xte, cols, seeds=(42, 0, 7, 13, 99), **kw)
sub = test[["cookie_id"]].merge(
    pd.DataFrame({"cookie_id": Xte.cookie_id, "score": p_test}),
    on="cookie_id",
    how="left",
)
sub["score"] = sub.score.clip(0, 1)
assert len(sub) == len(test)
assert sub.cookie_id.is_unique and sub.score.notna().all()
assert set(sub.cookie_id) == set(test.cookie_id)
assert sub.score.between(0, 1).all()
sub.to_csv("submission.csv", index=False, float_format="%.16f")
print(sub.shape, "mean", round(sub.score.mean(), 4), "seed", RANDOM_STATE)
sub.head()


(4909, 2) mean 0.0814 seed 42


,cookie_id,score
0,ck_315fb710a0e371e7,0.000073
1,ck_a76ee3b3e3e522fd,0.003908
2,ck_94c9a4d382689e82,0.003036
3,ck_8eaf9509ad9462a0,0.000066
4,ck_9a88a5a989cb5bc6,0.004087
